# Introduction

The notebook contains testing of a new approach of fitting orbitrap spectra and related time estimations and optimization.

The new approach is basically a different way to perform segmentation of spectrum prior to fitting.
The spectrum is splitted by only-sero-containing slices.


# Modules


In [ ]:
from lib.file_func import load_file
from lib.peak import detect_peaks
import numpy as np
import json
import matplotlib.pyplot as plt

# Instruments functions


In [ ]:
def resolution_function(mz):
    return 1.725e6 / np.sqrt(mz)


with open("C:/Users/alvai/Desktop/peak_shape.json", "r") as f:
    peak_shape = json.load(f)

# Existing fitting routine


In [ ]:
file_name = (
    "NewTestOrbitrap_20231227_1010_MION2_DBrMe_DHS_1ul_50pg_Explosive_Mix_60-600mz"
)
sample_file_data, all_peak_mzs = await detect_peaks(
    file_name,
    (peak_shape, resolution_function),
    # u_list = np.arange(50, 550, 1), # for testing
    if_exists="replace",
    dmz=0.5,
    return_peak_mzs=True,
)

# Separate window testing


In [ ]:
# Load spectrum
f = load_file(file_name)

In [ ]:
# mz value for testing
u = 366
# Window width
dmz = 0.5
# Get spectrum
mz = sample_file_data.mz
sum_spec = sample_file_data.signal.sum(dim="time").compute()
# Cut window from spectrum
mz_win = mz.sel(mz=slice(u - dmz, u + dmz)).compute().values
spec_win = sum_spec.sel(mz=slice(u - dmz, u + dmz)).compute().values

plt.plot(mz_win, spec_win)
plt.yscale("log")

## Time estimations


"stupid" solution time estimation


In [ ]:
%%time
def slow_segmentation(spec):
    indices = []
    ind_chunk = []
    for i, val in enumerate(spec):
        if val == 0:
            if ind_chunk == []:
                continue
            else:
                indices.append(np.array([ind_chunk[0] - 1] + ind_chunk + [ind_chunk[-1] + 1], dtype=np.int64))
                ind_chunk = []
        else:
            ind_chunk.append(i)
    return indices

slow_segmentation(sum_spec);

"wise" solution time estimation


In [ ]:
%%time
def fast_segmentation(spec):
    # Get non-zero indices
    non_zero_indices = np.flatnonzero(spec)
    # Split in chunks taking into account repeating zeros
    non_zero_indices = np.split(
        non_zero_indices, np.where(np.diff(non_zero_indices) > 1)[0] + 1
    )
    # Add zeros on chunk borders
    non_zero_indices = [
        np.concatenate(([chunk[0] - 1], chunk, [chunk[-1] + 1]))
        for chunk in non_zero_indices
    ]
    # Check wrong indices on the spectrum ends
    if non_zero_indices[0][0] < 0:
        # Remove negative index in the first chunk
        non_zero_indices[0] = non_zero_indices[0][1:]
    if non_zero_indices[-1][-1] == len(spec):
        # Remove out-of-range index from the last chunk
        non_zero_indices[-1][-1] = non_zero_indices[-1][:-1]
    # Remove short chunks
    non_zero_indices = [chunk for chunk in non_zero_indices if len(chunk) > 4]

    return non_zero_indices

fast_segmentation(sum_spec);

In [ ]:
slow_seg = slow_segmentation(spec_win)
fast_seg = fast_segmentation(spec_win)

fig, ax = plt.subplots(figsize=(15, 3), ncols=2)

ax[0].plot(mz_win, spec_win)
for i in slow_seg:
    ax[0].plot(mz_win[i], spec_win[i])
ax[0].set(yscale="log", title="Slow segmentation")


ax[1].plot(mz_win, spec_win)

for i in fast_seg:
    ax[1].plot(mz_win[i], spec_win[i])

ax[1].set(yscale="log", title="Fast segmentation")

fig.suptitle("Comparison of the outputs")

Check distribution of chunk lengths


In [ ]:
chunk_lengths = sorted([len(i) for i in fast_seg])
plt.plot(chunk_lengths)

# Notes

Peak "wings" are not fitted optimally, likely because we can not describe the whole range of intensities with one median peak shape.
One of the options could be to perform separate peak shape estimations for different intensity ranges.

The peak fitting is relatevely slow, 2 min for the 60...600 Th mz range.
That needs to be addressed at some point.

TOF spectra fitting routine must be improved.
Currently the only obvious thing is that baseline in spectrum should be taken into account.
